# Are the QC'd objects reproducible? A read-only check

Three sets of objects, all `qc_passed_{mouse,human}.h5ad`:

| name | what | made |
|---|---|---|
| `RECORD_QC` | the run of record: Cell Ranger -> CellBender -> QC -> concatenation | Nov 6, 2026 |
| `SENT` | `RECORD_QC` plus Nick's and Andy's position columns; the objects sent to Cagri | Nov 6, 2026 |
| `RERUN` | independent re-run of the whole pipeline from the FASTQs on different nodes | Nov 8, 2026 |

`RERUN/` holds the re-run's step-4 objects (as `RECORD_QC`) and `RERUN/with_coords/` its step-5 objects with
positions (as `SENT`).

Two claims are checked:

1. **Byte identity.** `RERUN`'s step-4 objects have the same sha256 as `RECORD_QC`'s.
2. **Dataset identity.** `RERUN/with_coords` equals `SENT` in every HDF5 group and dataset (nuclei, genes, counts,
   every obs column including positions, `obsm['spatial']`). The expected sole difference is the wording of one
   provenance note, `uns['coordinate_sources']`, which was reworded between Nov 6 and Nov 8. (With that note made
   identical the files are byte-identical too; see `diff_ready_objects.ipynb`.)

Plus a per-sample table of nuclei counts in all three sets.

This notebook only reads: h5py for the HDF5 contents, hashing for the checksums. It writes nothing. Kernel `rsc_uv`;
any compute node with 8 GB is enough (datasets are compared in slices); about 10 minutes, mostly checksumming
six 2.2 GB files.


In [1]:
import hashlib

import h5py
import numpy as np
import pandas as pd

NETSCRATCH = "/n/netscratch/eddy_lab/Everyone/nhilgert"
RECORD_QC = f"{NETSCRATCH}/2026_11_06_qc_regen"
SENT = f"{NETSCRATCH}/2026_11_06_qc_regen_nick_andy_coords"
RERUN = f"{NETSCRATCH}/2026_11_06_xmens_preprocess"
# sha256 values recorded when the objects were made (xmens_scripts/preprocess/record/objects_sha256.txt)
RECORDED_SHA256 = "/n/lab_storage/eddy_lab/users/nhilgert/xmens_scripts/preprocess/record/objects_sha256.txt"

SPECIES = ["mouse", "human"]
EXPECTED_NUCLEI = {"mouse": 69_307, "human": 70_149}


## 1. Byte identity: re-run step-4 objects vs the run of record

sha256 of the six files, next to the values recorded when the run of record was made.

In [2]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


recorded = {}
with open(RECORDED_SHA256) as fh:
    for line in fh:
        digest, path = line.split()
        recorded[path] = digest

rows = []
for species in SPECIES:
    record_file = f"{RECORD_QC}/qc_passed_{species}.h5ad"
    sent_file = f"{SENT}/qc_passed_{species}.h5ad"
    rerun_step4 = f"{RERUN}/qc_passed_{species}.h5ad"
    rerun_step5 = f"{RERUN}/with_coords/qc_passed_{species}.h5ad"
    files = [("record", record_file), ("sent", sent_file), ("rerun_step4", rerun_step4), ("rerun_step5", rerun_step5)]
    digests = {name: sha256(path) for name, path in files}
    rows.append({
        "species": species,
        "record == recorded value": digests["record"] == recorded.get(record_file),
        "sent == recorded value": digests["sent"] == recorded.get(sent_file),
        "rerun step 4 == record (byte-identical)": digests["rerun_step4"] == digests["record"],
        "rerun step 5 == sent (bytes)": digests["rerun_step5"] == digests["sent"],
        "sha256 record": digests["record"][:16] + "...",
        "sha256 rerun step 4": digests["rerun_step4"][:16] + "...",
    })
checksums = pd.DataFrame(rows).set_index("species")
checksums


,record == recorded value,sent == recorded value,rerun step 4 == record (byte-identical),rerun step 5 == sent (bytes),sha256 record,sha256 rerun step 4
species,,,,,,
mouse,True,True,True,False,fdae40f49d52ecb3...,fdae40f49d52ecb3...
human,True,True,True,False,a99d1b72d08f67f8...,a99d1b72d08f67f8...


Expected: the first three columns `True` for both species; the step-5 bytes `False` (the reworded note), which
section 2 resolves to the dataset level.

## 2. Dataset identity: every HDF5 group and dataset

`differences(a, b)` walks both files and compares the set of paths, every attribute, and every dataset's dtype,
shape and values (NaN equals NaN: unplaced nuclei have NaN positions in both). Datasets are read in slices so
memory stays small.

In [3]:
SLICE = 20_000_000  # elements per slice


def attrs_of(obj):
    return {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in obj.attrs.items()}


def walk(group, prefix=""):
    paths = {}
    for name, obj in group.items():
        path = f"{prefix}/{name}"
        paths[path] = "group" if isinstance(obj, h5py.Group) else "dataset"
        if isinstance(obj, h5py.Group):
            paths.update(walk(obj, path))
    return paths


def datasets_equal(a, b):
    if a.dtype != b.dtype:
        return False, f"dtype {a.dtype} vs {b.dtype}"
    if a.shape != b.shape:
        return False, f"shape {a.shape} vs {b.shape}"
    if a.size == 0:
        return True, ""
    if a.ndim == 0:
        va, vb = a[()], b[()]
        return bool(va == vb), f"{va!r} vs {vb!r}"
    nan_ok = np.issubdtype(a.dtype, np.floating)
    rows_per_slice = max(1, SLICE // int(np.prod(a.shape[1:]))) if a.ndim > 1 else SLICE
    for start in range(0, a.shape[0], rows_per_slice):
        stop = min(start + rows_per_slice, a.shape[0])
        if not np.array_equal(a[start:stop], b[start:stop], equal_nan=nan_ok):
            return False, f"values differ in [{start}:{stop})"
    return True, ""


def differences(file_a, file_b):
    """List of (path, what differs) between two .h5ad files; empty when they hold the same object."""
    found = []
    with h5py.File(file_a, "r") as fa, h5py.File(file_b, "r") as fb:
        paths_a, paths_b = walk(fa), walk(fb)
        for path in sorted(set(paths_a) - set(paths_b)):
            found.append((path, "only in A"))
        for path in sorted(set(paths_b) - set(paths_a)):
            found.append((path, "only in B"))
        for path in sorted(set(paths_a) & set(paths_b)):
            if paths_a[path] != paths_b[path]:
                found.append((path, f"{paths_a[path]} in A, {paths_b[path]} in B"))
                continue
            if attrs_of(fa[path]) != attrs_of(fb[path]):
                found.append((path, "attributes differ"))
            if paths_a[path] == "dataset":
                equal, note = datasets_equal(fa[path], fb[path])
                if not equal:
                    found.append((path, note))
        n_compared = len(set(paths_a) & set(paths_b))
    return found, n_compared


report = []
for species in SPECIES:
    pairs = {
        "rerun step 4 vs RECORD_QC": (f"{RERUN}/qc_passed_{species}.h5ad", f"{RECORD_QC}/qc_passed_{species}.h5ad"),
        "rerun step 5 vs SENT": (f"{RERUN}/with_coords/qc_passed_{species}.h5ad", f"{SENT}/qc_passed_{species}.h5ad"),
    }
    for label, (a, b) in pairs.items():
        found, n_compared = differences(a, b)
        report.append({"species": species, "comparison": label, "paths compared": n_compared,
                       "differences": len(found),
                       "which": "; ".join(f"{p}: {w}" for p, w in found) if found else "none"})
dataset_report = pd.DataFrame(report).set_index(["species", "comparison"])
pd.set_option("display.max_colwidth", 400)
dataset_report


paths compared  differences  \
species comparison                                               
mouse   rerun step 4 vs RECORD_QC              49            0   
        rerun step 5 vs SENT                   57            1   
human   rerun step 4 vs RECORD_QC              49            0   
        rerun step 5 vs SENT                   57            1   

                                                                                                                                                                                                                                                                                                                                                                                                                                             which  
species comparison                                                                                                                                                                                                                                                                                                                                                                                                                                  
mouse   rerun step 4 vs RECORD_QC                                                                                                                                                                                                                                                                                                                                                                                                             none  
        rerun step 5 vs SENT       /uns/coordinate_sources/obs x_um_dbscan_nick, y_um_dbscan_nick (same values: obs x_um_dbscan, y_um_dbscan; obsm spatial): b"Nick's March 2026 Slide-tags run: /n/lab_storage/eddy_lab/users/nhilgert/Slide-tags/coords/<lane>_coords.csv, columns dbscan_1, dbscan_2, attached by ingest.py" vs b"March 2026 Slide-tags re-run: /n/lab_storage/eddy_lab/users/nhilgert/Slide-tags/coords/<lane>_coords.csv, ...  
human   rerun step 4 vs RECORD_QC                                                                                                                                                                                                                                                                                                                                                                                                             none  
        rerun step 5 vs SENT       /uns/coordinate_sources/obs x_um_dbscan_nick, y_um_dbscan_nick (same values: obs x_um_dbscan, y_um_dbscan; obsm spatial): b"Nick's March 2026 Slide-tags run: /n/lab_storage/eddy_lab/users/nhilgert/Slide-tags/coords/<lane>_coords.csv, columns dbscan_1, dbscan_2, attached by ingest.py" vs b"March 2026 Slide-tags re-run: /n/lab_storage/eddy_lab/users/nhilgert/Slide-tags/coords/<lane>_coords.csv, ...

Expected: 0 differences for step 4 vs the record (49 paths each); exactly 1 for step 5 vs SENT (57 paths each),
at `/uns/coordinate_sources/...`, whose two values are the Nov 6 and Nov 8 wordings of the same note.

## 3. Nuclei per sample in all three sets

Read from `obs/sample` only (h5py), so it is fast. The three columns must agree and sum to the totals in the
Methods (69,307 mouse, 70,149 human).

In [4]:
def nuclei_per_sample(path):
    """Counts per sample from the categorical obs/sample column, without loading the object."""
    with h5py.File(path, "r") as fh:
        column = fh["obs/sample"]
        categories = column["categories"][()].astype(str)
        codes = column["codes"][()]
    counts = pd.Series(codes).value_counts().sort_index()
    return pd.Series(counts.values, index=categories[counts.index], name="n")


tables = []
for species in SPECIES:
    table = pd.DataFrame({
        "record": nuclei_per_sample(f"{RECORD_QC}/qc_passed_{species}.h5ad"),
        "sent": nuclei_per_sample(f"{SENT}/qc_passed_{species}.h5ad"),
        "rerun": nuclei_per_sample(f"{RERUN}/with_coords/qc_passed_{species}.h5ad"),
    })
    table["all equal"] = table.nunique(axis=1) == 1
    table.loc["total"] = [table["record"].sum(), table["sent"].sum(), table["rerun"].sum(), table["all equal"].all()]
    table.index = pd.MultiIndex.from_product([[species], table.index])
    tables.append(table)
    assert table.loc[(species, "total"), "rerun"] == EXPECTED_NUCLEI[species], species
per_sample = pd.concat(tables)
per_sample


record   sent  rerun  all equal
mouse 1_dpi_early_mck1         3497   3497   3497       True
      1_dpi_late_mck12a        7768   7768   7768       True
      1_dpi_late_mck12b        7037   7037   7037       True
      2_dpi_mck15              5153   5153   5153       True
      3_dpi_mck10a             8407   8407   8407       True
      3_dpi_mck10b             8246   8246   8246       True
      4_dpi_bleeding_mck11a    8746   8746   8746       True
      uninduced_mck8a         12027  12027  12027       True
      uninduced_mck8b          8426   8426   8426       True
      total                   69307  69307  69307       True
human menstrual_mck5a         12742  12742  12742       True
      menstrual_mck5b         12634  12634  12634       True
      menstrual_mck5c         11625  11625  11625       True
      secretory_mck6a         12983  12983  12983       True
      secretory_mck6b         12696  12696  12696       True
      secretory_mck6c          7469   7469   7469       True
      total                   70149  70149  70149       True

## 4. Verdict

In [5]:
byte_ok = checksums["rerun step 4 == record (byte-identical)"].all()
step4_ok = (dataset_report.xs("rerun step 4 vs RECORD_QC", level="comparison")["differences"] == 0).all()
step5 = dataset_report.xs("rerun step 5 vs SENT", level="comparison")
step5_ok = (step5["differences"] == 1).all() and step5["which"].str.contains("coordinate_sources").all()
counts_ok = per_sample["all equal"].all()

print(f"step-4 objects byte-identical to the run of record:          {byte_ok}")
print(f"step-4 objects identical in every dataset:                    {step4_ok}")
print(f"step-5 objects equal SENT except the provenance note wording: {step5_ok}")
print(f"nuclei per sample identical in record, SENT and re-run:       {counts_ok}")
print()
print("REPRODUCED" if all([byte_ok, step4_ok, step5_ok, counts_ok]) else "NOT REPRODUCED: see the tables above")


step-4 objects byte-identical to the run of record:          True
step-4 objects identical in every dataset:                    True
step-5 objects equal SENT except the provenance note wording: True
nuclei per sample identical in record, SENT and re-run:       True

REPRODUCED
